In [1]:
%matplotlib inline
import os
import sys
import glob
import numpy as np
import pandas as pd
from datetime import datetime
import xarray as xr
from multiprocessing import Pool
import multiprocessing, logging
logger = multiprocessing.log_to_stderr()
logger.setLevel(multiprocessing.SUBDEBUG)

from read_deterministic_data import load_GFS_datasets, load_ECMWF_datasets, read_preprocessed_IVT_data
from calc_funcs import format_timedelta_to_HHMMSS
from cw3e_tools import remove_tmp_data_files
from plotter import plot_time_height_meteograms

model_name = 'ECMWF' ## GFS or ECMWF
print('Creating WV Flux Meteograms for {0}'.format(model_name))

## FIRST LOOP - LEAD TIME ##
if model_name == 'ECMWF':
    arr1 = np.arange(0, 72+3, 3)
    arr2 = np.arange(78, 168+6, 6)
    F_lst = np.concatenate((arr1, arr2), axis=0)
elif model_name == 'GFS':
    F_lst = np.arange(0, 168+3, 3)

fdate='2025021112' ## set this to None to get most recently downloaded data

ERROR 1: PROJ: proj_create_from_database: Open of /home/dnash/miniconda3/envs/SEAK-impacts/share/proj failed


Creating WV Flux Meteograms for ECMWF


In [2]:
#################################
### CHECK TO REMOVE TMP FILES ###
#################################
print('Removing tmp intermediate data files...') 
# Specify the directory and the pattern
tmp_directory = "/data/projects/operations/wvflux_meteograms/data/tmp/"
pattern = "tmp_{0}*.nc".format(model_name)  # Delete all tmp files
remove_tmp_data_files(tmp_directory, pattern)

Removing tmp intermediate data files...


In [3]:
# ### 000 - prate
# ### 003 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp
# ### 006 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp

# filename = '/data/projects/external_datasets/GFS/processed/{0}/{0}_F009.grb2'.format(fdate)
# ds = xr.open_dataset(filename,
#                         engine='cfgrib',backend_kwargs={"indexpath": ''}, filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'})
# ds

In [5]:
%%time
####################################
### TEST WITH A FEW NON-PARALLEL ###
####################################
for i, F in enumerate(F_lst):
    start_time = pd.Timestamp.today()
    #######################
    ### LOAD MODEL DATA ###
    #######################
    print('... Loading {0} data for {1} hour lead'.format(model_name, F))
    if model_name == 'ECMWF':
        s = load_ECMWF_datasets(F=F, fdate=fdate)
        model_data= s.calc_vars()

    elif model_name == 'GFS':
        s = load_GFS_datasets(F=F, fdate=fdate)
        model_data = s.calc_vars()
        
model_data

... Loading ECMWF data for 0 hour lead
2025021112
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S2D02111200021112001.grb
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S1D02111200021112001
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S1D02111200021115001
... Loading ECMWF data for 3 hour lead
2025021112
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S2D02111200021115001.grb
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S1D02111200021115001
... Loading ECMWF data for 6 hour lead
2025021112
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S2D02111200021118001.grb
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S1D02111200021118001
... Loading ECMWF data for 9 hour lead
2025021112
[INFO] reading ECMWF file: /data/downloaded/Forecasts/ECMWF/NRT_data/2025021112/S


KeyboardInterrupt



In [5]:
# ############################################
# ### PREPROCESS INTERMEDIATE GFS OR ECMWF ###
# ############################################
# print('...preprocess intermediate data ...')

# def multiP_preprocess_intermediate(model_name, F, fdate):
#     ##############################
#     ### PREP INTERMEDIATE DATA ###
#     ##############################
#     print('... Loading data for {0} hour lead'.format(F))
#     if model_name == 'ECMWF':
#         s = load_ECMWF_datasets(F=F, fdate=fdate)
#         model_data = s.calc_vars()

#     elif model_name == 'GFS':
#         s = load_GFS_datasets(F=F, fdate=fdate)
#         model_data, tmp = s.calc_vars()

# if __name__ == '__main__':
#     with Pool(processes=30) as pool:
#         for F in F_lst:
#             result = pool.apply_async(func=multiP_preprocess_intermediate,args=(model_name, F, fdate))
#         pool.close()
#         pool.join()

In [6]:
##############################
### LOAD INTERMEDIATE DATA ###
##############################
# tmp_directory = "/home/dnash/comet_data/tmp/"
tmp_directory = "/data/projects/operations/wvflux_meteograms/data/tmp/"
files = glob.glob(tmp_directory + 'tmp_{0}_*.nc'.format(model_name))
sorted_files = sorted(files)
ds = xr.open_mfdataset(sorted_files, engine='netcdf4', concat_dim="step", combine='nested')

## need to fix precipitation
if model_name == 'ECMWF':
    tp = ds.tp.fillna(0) # make sure nan at beginning is 0 to do calculation right
    tp = tp.diff(dim='step') # calculate difference between time steps
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, tp]) # merge dataset with new tp
    
if model_name == 'GFS':
    ts_3hr = pd.timedelta_range(start='0 day', periods=57, freq='3H')
    ts_6hr = pd.timedelta_range(start='0 day', periods=29, freq='6H')
    tp = ds.tp ## pull out tp
    prec_3hr = tp.sel(step=ts_3hr[1::2]) ## grab only the 3hr values
    tp2 = tp.diff(dim='step') ## calculate difference in precip
    ## the values for 6hr timesteps are correct, the values for 3hr timesteps are incorrect
    prec_6hr = tp2.sel(step=ts_6hr[1:]) # grab only the 6hr values
    new_prec = prec_3hr.combine_first(prec_6hr) # combine the correct 3hr values with the correct 6hr values
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, new_prec]) # merge dataset with new tp
    
ivt = read_preprocessed_IVT_data(model=model_name, F_lst=F_lst, fdate=pd.to_datetime(ds.time.values).strftime('%Y%m%d%H'))
ds = ds.assign(ivt=(['step','latitude','longitude'],ivt.ivt.values))

/home/dnash/miniconda3/envs/SEAK-impacts/lib/python3.12/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
Struct() takes at most 1 argument (3 given)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


In [3]:
import numpy as np
lat_lst = np.arange(26, 51, 1)
lon_lst = np.arange(111, 128, 1)
var_lst = ['r', 'wvflux']
dur_lst = [7, 3]

arglst_USWEST = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                arglst_USWEST.append((x,y, varname, dur))

lat_lst = np.arange(51, 70, 1)
lon_lst = np.arange(130, 175, 1)               
arglst_AK = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                arglst_AK.append((x,y, varname, dur))
                
final_arglst = arglst_USWEST + arglst_AK

for i, argval in enumerate(final_arglst[:2]):
    lon, lat, varname, dur = argval
    print(lon, lat, varname, dur)

111 26 r 7
111 26 r 3


In [11]:
len(arglst_AK)

3420

In [2]:
len(final_arglst)

5120

In [ ]:
print('...create plots ...')

def multiP_create_time_height_meteograms(ds, varname, lat, lon, model_name, duration):
    plot_time_height_meteograms(ds, varname, lat, 360-lon, model_name, duration)

if __name__ == '__main__':
    with Pool(processes=30) as pool:
        for argval in final_arglst[60:90]:
            lon, lat, varname, dur = argval
            result = pool.apply_async(func=multiP_create_time_height_meteograms,args=(ds, varname, lat, lon, model_name, dur))
        pool.close()
        pool.join()

[DEBUG/MainProcess] created semlock with handle 140275073884160
[DEBUG/MainProcess] created semlock with handle 140275073880064
[DEBUG/MainProcess] created semlock with handle 140275073875968
[DEBUG/MainProcess] created semlock with handle 140274772738048
[DEBUG/MainProcess] created semlock with handle 140274173358080
[DEBUG/MainProcess] created semlock with handle 140274173353984


...create plots ...


[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-1] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-2] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-3] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-4] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-5] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-6] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-7] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-8] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-9] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-10] child process calling self.run()
[DEBUG/MainProcess] added worker
[INFO/ForkPoolWorker-11] child process calling self.run()
[DEBUG/M

In [ ]:
# %%time
# for argval in final_arglst[0:30]:
#     lon, lat, varname, dur = argval
#     plot_time_height_meteograms(ds, varname, lat, 360-lon, model_name, dur)